# 09 — Passing-Network Vulnerability: Part 2, Single-Player Removal Simulation

*2018 & 2022 FIFA World Cup · StatsBomb event data*

**Central question:** for each team-match passing network, if one player is removed,
how much does the network structurally deteriorate relative to its original state?

**Scope:** this notebook does Part 2 only — single-player removal. It does not do
Monte Carlo random-removal comparison, two-player combination search, team robustness
rankings, or outcome analysis (Parts 3-5). It builds directly on the validated Part 1
baseline (`notebooks/08_player_disruption_baseline.ipynb`).

**What a damage score does *not* mean:** "if this player were absent, the team would
lose X% of its passing ability." **What it does mean:** "X% of the network's *observed*
structural capacity is directly attributable to this player's position and connections."
This is a structural disruption simulation on the graph as recorded — it does not model
how teammates would adapt if a player actually left the pitch.

In [1]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import network_vulnerability as nv

PROC_DIR = Path('../data/processed')

player_network_baseline = pd.read_csv(PROC_DIR / 'player_network_baseline.csv')

removal_path = PROC_DIR / 'single_player_removal.csv'
if removal_path.exists():
    single_player_removal = pd.read_csv(removal_path)
else:
    single_player_removal = nv.simulate_single_player_removal(player_network_baseline)
    single_player_removal.to_csv(removal_path, index=False)

print(single_player_removal.shape)

(3758, 32)


## Validation checks

In [2]:
nv.run_removal_validation_checks(single_player_removal)

SINGLE-PLAYER REMOVAL — VALIDATION CHECKS

1. Removing a player always decreases node count by exactly 1: True
2. Removed edge count matches incident-edge count: 3758 / 3758 rows


3. Spot-checked 25 removals: 0 still contained the removed player
4. Infinities in damage columns: {'density_damage': 0, 'edge_damage': 0, 'largest_component_damage': 0, 'efficiency_damage': 0, 'progressive_capacity_damage': 0}

5. Negative damage values (metric improved after removal — not automatically an error):
   density_damage: 1665 / 3758 (44.3%)
   edge_damage: 0 / 3758 (0.0%)
   largest_component_damage: 0 / 3758 (0.0%)
   efficiency_damage: 1631 / 3758 (43.4%)
   progressive_capacity_damage: 0 / 3758 (0.0%)
   Density in particular can rise after removing a low-degree player because
   the n*(n-1) denominator shrinks faster than the edge count.

6. Total player-removal simulations: 3758
   Team-matches covered: 256
   Average simulations per team-match: 14.68

   Damage metric distributions:
       density_damage  edge_damage  largest_component_damage  \
count       3758.0000    3758.0000                 3758.0000   
mean          -0.0000       0.1362                    0.000

## Sanity checks on the removal simulation

### Shape and first 10 rows

In [3]:
print(f"Shape: {single_player_removal.shape}")
display_cols = [
    'match_id', 'team', 'removed_player', 'total_pass_involvement',
    'low_involvement_flag', 'density_damage', 'edge_damage',
    'largest_component_damage', 'efficiency_damage', 'progressive_capacity_damage',
]
single_player_removal[display_cols].head(10)

Shape: (3758, 32)


,match_id,team,removed_player,total_pass_involvement,low_involvement_flag,density_damage,edge_damage,largest_component_damage,efficiency_damage,progressive_capacity_damage
0,3857254,Denmark,Kasper Dolberg,26,False,-0.032878,0.104839,0.0,-0.005587,0.086957
1,3857254,Denmark,Christian Dannemann Eriksen,133,False,0.088089,0.209677,0.0,0.026644,0.217391
2,3857254,Denmark,Andreas Skov Olsen,49,False,-0.014268,0.120968,0.0,0.000859,0.136646
3,3857254,Denmark,Rasmus Nissen Kristensen,83,False,-0.014268,0.120968,0.0,-0.005587,0.192547
4,3857254,Denmark,Joakim Mæhle,83,False,0.032258,0.161290,0.0,0.020198,0.229814
5,3857254,Denmark,Pierre-Emile Højbjerg,136,False,0.060174,0.185484,0.0,0.020198,0.211180
6,3857254,Denmark,Joachim Andersen,120,False,0.078784,0.201613,0.0,0.026644,0.142857
7,3857254,Denmark,Kasper Schmeichel,35,False,-0.023573,0.112903,0.0,-0.005587,0.074534
8,3857254,Denmark,Andreas Christensen,151,False,0.013648,0.145161,0.0,0.000859,0.167702
9,3857254,Denmark,Simon Thorup Kjær,95,False,0.004342,0.137097,0.0,0.000859,0.118012


### 3. Top 3 players by efficiency damage and progressive-capacity damage, for 5 example team-matches

Deliberately not filtering out low-involvement players here — the point of this check is
to see whether the two damage metrics agree with each other and with raw involvement, not
to draw conclusions yet.

In [4]:
example_keys = (
    single_player_removal[['match_id', 'team']]
    .drop_duplicates()
    .sample(5, random_state=7)
    .itertuples(index=False)
)

for match_id, team in example_keys:
    sub = single_player_removal[
        (single_player_removal['match_id'] == match_id) & (single_player_removal['team'] == team)
    ]
    print(f"=== match {match_id} — {team} ===")
    print("Top 3 by efficiency damage:")
    print(sub.sort_values('efficiency_damage', ascending=False)
          [['removed_player', 'total_pass_involvement', 'efficiency_damage']].head(3)
          .to_string(index=False))
    print("Top 3 by progressive-capacity damage:")
    print(sub.sort_values('progressive_capacity_damage', ascending=False)
          [['removed_player', 'total_pass_involvement', 'progressive_capacity_damage']].head(3)
          .to_string(index=False))
    print()

=== match 7530 — Australia ===
Top 3 by efficiency damage:
 removed_player  total_pass_involvement  efficiency_damage
    Josh Risdon                      57           0.040016
     Aaron Mooy                      86           0.023699
Trent Sainsbury                      87           0.012821
Top 3 by progressive-capacity damage:
     removed_player  total_pass_involvement  progressive_capacity_damage
Aziz Eraltay Behich                      91                     0.281818
        Josh Risdon                      57                     0.263636
      Mark Milligan                     133                     0.263636

=== match 8655 — France ===
Top 3 by efficiency damage:
removed_player  total_pass_involvement  efficiency_damage
  N'Golo Kanté                      73           0.128217
    Paul Pogba                      60           0.020779
Olivier Giroud                      35           0.020779
Top 3 by progressive-capacity damage:
      removed_player  total_pass_involvement  pr

### 4. Does removal damage track ordinary involvement/centrality, or add information?

The question isn't whether highly-involved players are "the most vulnerable" — it's
whether the simulation tells us something involvement or centrality alone wouldn't.

In [5]:
part1_betweenness = player_network_baseline[['match_id', 'team', 'player', 'betweenness_centrality']].rename(
    columns={'player': 'removed_player'}
)
merged = single_player_removal.merge(part1_betweenness, on=['match_id', 'team', 'removed_player'], how='left')

corr_involvement_eff = merged['total_pass_involvement'].corr(merged['efficiency_damage'])
corr_involvement_prog = merged['total_pass_involvement'].corr(merged['progressive_capacity_damage'])
corr_betweenness_eff = merged['betweenness_centrality'].corr(merged['efficiency_damage'])

print(f"total_pass_involvement vs efficiency_damage        : r = {corr_involvement_eff:.3f}")
print(f"total_pass_involvement vs progressive_capacity_damage: r = {corr_involvement_prog:.3f}")
print(f"betweenness_centrality (Part 1) vs efficiency_damage : r = {corr_betweenness_eff:.3f}")

total_pass_involvement vs efficiency_damage        : r = 0.541
total_pass_involvement vs progressive_capacity_damage: r = 0.684
betweenness_centrality (Part 1) vs efficiency_damage : r = 0.371


## Next step

Part 2 output is cached to `data/processed/single_player_removal.csv`. Part 3 (targeted
vs. random disruption via Monte Carlo) will load this directly.